# Bayesian Hierarhical Modelling of Categorical variables

To improve tabular data models beydong standard encoding methods.

Usefull when categorical data has
- many categories
- imbalanced catagories
- small/medium sized data
- tabular data

Hierarchical models are essentially statistical models that account for data that is grouped or nested. These models allow us to pool information across groups while still permitting group-specific effects.

---

Standard approaches and problems with categorical variables are:
- One-hot encoding: High dimensionality and overfitting (intracorrelation too?)
- Target encoding: Information leakage and unstable for rare categories
- Embeddings: Needs large amounts of data

---

Bayesian hierarchical modelling of categorical variables can be described as partial pooling to build better encodings for high-cardinality caterogicals. Bayesian hierarchical modelling improves handling of categorical features through partial pooling — instead of treating each category independently (no pooling) or ignoring category identity entirely (full pooling), it shrinks per-category estimates toward a global mean. This is especially powerful for rare or unseen categories.

For a categorical feature like city with many levels, instead of one-hot encoding (which gives each city a fully independent coefficient), BHM estimates:
- A global mean effect across all cities
- Per-city offsets shrunk toward that mean, proportional to how much data each city has

Rare cities get pulled strongly toward the global mean; common cities retain their own estimates.

Bayesian hierarchical modelling treats each category's effect as drawn from a shared distribution. 
$$
\theta \sim \mathcal{N}(\mu, \tau^2)
$$
for category $j$. Each category has its own parameter, but they are not independent and the come from a common distribution. 

---

**The Problem with Standard Encodings**

With one-hot encoding, logistic regression estimates a completely independent coefficient for each category level. For a category with $n_c$ observations:
$$
\hat{\beta}_c^{\text{MLE}} = \underset{\beta_c}{\arg\max} \prod_{i: x_i = c} p(y_i | \beta_c)
$$
This is no-pooling: each group's estimate is determined solely by its own data. For rare categories, the MLE is unreliable — high variance, prone to overfitting.

---

**The Generative Model**

Bayesian hierarchical modelling posits that category-level parameters are drawn from a shared population distribution, rather than being fixed independent constants. For a binary target $y_i \in \{0, 1 \}$, observation $i$ belonging to category $c$ is sampled from a Bernoulli distribution parametrized by $\theta_c$, where $\theta_c$ is a shared prior across all categories sampled from a Beta distribution with parameters $\alpha$ and $\beta$. 

The key insights are $\theta_c$ are random variables, not fixed unknowns. These parameters are linked via the shared prior, creating the hierarchical structure. The full joint distribution is 
$$
p(\theta_1, \dots, \theta_C, \alpha, \beta \mid \mathbf{y}) \propto p(\alpha, \beta) \prod_{c=1}^C p(\theta_c \mid \alpha, \beta) \prod_{i : x_i = c} p(y_i \mid \theta_c)
$$
Here, $p(\theta_c \mid \alpha, \beta)$ is the prior and $p(y_i \mid \theta_c)$ is the likelihood.

---

**Conjugate posterior**

The Beta distribution is conjugate to the Bernoulli likelihood, meaning the posterior is analytically tractable. For category $c$ with $n_c$ observations and $s_c = \sum_{i : x_i = c} y_i$ successes:
$$
p(\theta_c \mid \mathbf{y}_c, \alpha, \beta) = Beta(\alpha + s_c, \beta + n_c - s_c)
$$
The posterior mean - i.e., the point estimate for $\theta_c$ - is:
$$
\mathbb{E} [ \theta_c \mid \mathbf{y}_c ] = \frac{\alpha + s_c}{\alpha + \beta + n_c}
$$
This can be rewritten as a weighted average between the local MLE and the prior mean
$$
\mathbb{E} [ \theta_c \mid \mathbf{y}_c ] = \frac{n_c}{n_c + k} \cdot \frac{s_c}{n_c} + \frac{k}{n_c + k} \cdot \mu
$$
using
$$
\mu = \frac{\alpha}{\alpha + \beta}
$$
and where $k = \alpha + \beta$ is the prior strength (equiv. to the number of pseudo-observations added from the prior). This gives a smoothing formula. 

---

**Partial pooling as a spectrum**

The shrinkage weight 
$$
\lambda_c = \frac{n_c}{n_c + k}
$$
tells you how much each cataegory trusts its own data:
$$
\lambda_c = \begin{cases}\rightarrow 1 & \text{if}\; n_c \gg k \; \text{(large group - trust local data)} \\ \rightarrow 0 & \text{if}\; n_c \ll k \; \text{(small group - shrink to global mean)} \end{cases}
$$

This defines a spectrum between two extremes:
- No pooling (one-hot): $\hat{\theta}_c = s_c/n_c$. Assumed groups are completely independent.
- Partial pooling (Bayesian hierarchical): $\hat{\theta}_c = \lambda_c \hat{\mu} + (1 - \lambda_c) \mu$. Groups share a population distribution.
- Full pooling: $\hat{\theta}_c = \mu \forall c$. Category identity is irrelevant.

---

**Connection to regularisation**

The posterior mean estimator is equivalent to a penalized MLE. Maximizing the log-posterior:
$$
\log p(\theta_c \mid \mathbf{y}_c) = \sum_{i : x_i = c} \log p(y_i \mid \theta_c) + \log p(\theta_c \mid, \alpha, \beta)
$$
The last term is the log-prior, acting as a regulariser penalising $\theta_c$ for deviating from the population mean $\mu$. This is the Bayesian analouge for L2 (Ridge) regularisation - except L2 shrinks toward zero, while Bayesian shrinks towards the data-informed global mean. 

---

**Hyperparameters and empirical Bayes**

In a full Bayesian treatment, you would place a hyperprioer on $\alpha$, $\beta$ and integrate them out. In the code, we instead use empirical Bayes (Type-II MLE): estimate $\mu$ from the data as the global target mean $\bar{y}$, and treat $k = \alpha + \beta$ as a tuneable hyperparameter
$$
\hat{mu} = \bar{y} = \frac{1}{n} \sum_i y_i, \quad k \; \text{tuned by cross-validation}
$$
This avoids MCMC while retaining the shrinkage benefit. The is that the uncertainty in the hyperparameters themselves are ignored.

---

**What the logistic regression layer then does**

After encoding the feature vector for observation $i$ becomes
$$
\mathbf{\tilde{x}}_i = \left[ x_i^{(1)}, x_i^{(2)}, \hat{\theta}_{c1(i)}, \hat{\theta}_{c2(i)} \right ],
$$
where the two first entries are continuous features and the last two are the Bayesian-smoothed encodings. Logistic regression the models:
$$
p(y_i = 1 \mid \mathbf{\tilde{x}}_i) = \sigma \left( \beta_0 + \beta_1 x_i^{(1)} + \beta_2 x_i^{(2)} + \beta_3 \hat{\theta}_{c1(i)}+ \beta_4 \hat{\theta}_{c2(i)} \right)
$$
The smoothed encodings enter as continuous features, making the categorical information numerically stable and carrying the hierarchical structure learned in step 1.

In [2]:
import numpy as np 
import pandas as pd 
np.random.seed(42)

n = 500

df = pd.DataFrame({
    'cont_1':  np.random.randn(n),
    'cont_2':  np.random.randn(n),
    'cat_1':   np.random.choice(['A', 'B', 'C', 'D'], n, p=[0.5, 0.3, 0.15, 0.05]),
    'cat_2':   np.random.choice(['X', 'Y', 'Z'],      n),
})

# Target influenced by category (A and X have higher probability)
log_odds = (
    0.5 * df['cont_1']
    - 0.3 * df['cont_2']
    + df['cat_1'].map({'A': 1.0, 'B': 0.0, 'C': -0.5, 'D': -1.0})
    + df['cat_2'].map({'X': 0.8, 'Y': 0.0, 'Z': -0.8})
)
df['target'] = (np.random.rand(n) < 1 / (1 + np.exp(-log_odds))).astype(int)

X = df[['cont_1', 'cont_2', 'cat_1', 'cat_2']]
y = df['target']

X

,cont_1,cont_2,cat_1,cat_2
0,0.496714,0.926178,A,Z
1,-0.138264,1.909417,A,X
2,0.647689,-1.398568,B,X
3,1.523030,0.562969,B,Z
4,-0.234153,-0.650643,A,Z
...,...,...,...,...
495,0.538910,-0.281100,B,X
496,-1.037246,1.797687,B,X
497,-0.190339,0.640843,B,X
498,-0.875618,-0.571179,A,Y


In [11]:
X["cat_1"].value_counts()

cat_1
A    246
B    152
C     71
D     31
Name: count, dtype: int64

In [12]:
X["cat_2"].value_counts()

cat_2
Y    177
X    163
Z    160
Name: count, dtype: int64

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class BayesianTargetEncoder(BaseEstimator, TransformerMixin):
    """
    Replaces each category with a smoothed posterior mean of the target.
    
    For category c with n_c observations and local mean y_c:
        encoded = (n_c * y_c + k * global_mean) / (n_c + k)
    
    k is the 'prior strength' — higher k = more shrinkage toward global mean.
    This is a hyperparameter for tuning.
    Equivalent to a Beta-Binomial conjugate model for binary targets.
    """
    def __init__(self, cols, prior_strength=10):
        self.cols = cols          # list of categorical column names
        self.prior_strength = prior_strength
    
    def fit(self, X, y):
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        y = np.array(y)
        
        self.global_mean_ = y.mean()
        self.encodings_ = {}
        
        for col in self.cols:
            stats = (
                pd.DataFrame({'cat': X[col], 'target': y})
                .groupby('cat')['target']
                .agg(['mean', 'count'])
            )
            k = self.prior_strength
            g = self.global_mean_
            
            # Posterior mean: weight local evidence vs prior
            stats['encoded'] = (
                (stats['count'] * stats['mean'] + k * g) /
                (stats['count'] + k)
            )
            self.encodings_[col] = stats['encoded'].to_dict()
        
        return self
    
    def transform(self, X):
        X = pd.DataFrame(X) if not isinstance(X, pd.DataFrame) else X.copy()
        
        for col in self.cols:
            # Unseen categories fall back to global mean — no more KeyErrors
            X[col] = X[col].map(self.encodings_[col]).fillna(self.global_mean_)
        
        return X.values

In [4]:
# ── Example usage ──────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipeline = Pipeline([
    ('encoder', BayesianTargetEncoder(cols=['cat_1', 'cat_2'], prior_strength=10)),
    ('scaler',  StandardScaler()),
    ('logreg',  LogisticRegression(C=1.0, random_state=42)),
])

pipeline.fit(X, y)
print("Pipeline fitted successfully.")
print("Learned encodings for cat_1:", pipeline['encoder'].encodings_['cat_1'])

Pipeline fitted successfully.
Learned encodings for cat_1: {'A': 0.678671875, 'B': 0.5292592592592592, 'C': 0.42888888888888893, 'D': 0.3839024390243902}


In [9]:
X["cat_1"].unique(), pipeline['encoder'].encodings_['cat_1']

(<StringArray>
 ['A', 'B', 'C', 'D']
 Length: 4, dtype: str,
 {'A': 0.678671875,
  'B': 0.5292592592592592,
  'C': 0.42888888888888893,
  'D': 0.3839024390243902})

In [10]:
X["cat_2"].unique(), pipeline['encoder'].encodings_['cat_2']

(<StringArray>
 ['Z', 'X', 'Y']
 Length: 3, dtype: str,
 {'X': 0.7326011560693643, 'Y': 0.5814973262032085, 'Z': 0.4043529411764706})